In [117]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [118]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [119]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [120]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

### Aggregation

In [121]:
data_query = f"""
    select * from TRN_MIL_DF_HEURISTICS_OUTPUT 
"""
df_heuristics = pd.read_sql(data_query, dev_conn)
df_heuristics

,CHANNEL,PORTFOLIO,BRAND,BRAND CLASS,RUN_MONTH,M MONTH,MONTH,ASM,DEPOT,PSKU,PROPHET VOL,RF_VOL,PROPHET HEURISTIC VOL,RF HEURISTIC VOL
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.700,5.264959,4.500
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.600,4.547336,4.500
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.700,7.053803,4.500
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.800,5.662848,4.500
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.700,4.500000,4.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+1,2025-11-30,QCW2,D461,718836,3.767871,4.908,5.040000,5.040
282200,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+2,2025-12-31,QCW2,D461,718836,4.907447,4.932,5.040000,5.040
282201,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+3,2026-01-31,QCW2,D461,718836,4.751205,4.260,5.040000,5.040
282202,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+4,2026-02-28,QCW2,D461,718836,8.038001,4.776,8.038001,5.040


In [122]:
df_heuristics.columns = df_heuristics.columns.str.lower()
df_heuristics['run_month'] = pd.to_datetime(df_heuristics['run_month'])
df_heuristics['month'] = pd.to_datetime(df_heuristics['month'])
df_heuristics


,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M,2025-11-30,BCE1,D231,718589,5.264959,2.700,5.264959,4.500
1,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+1,2025-12-31,BCE1,D231,718589,4.547336,3.600,4.547336,4.500
2,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+2,2026-01-31,BCE1,D231,718589,7.053803,2.700,7.053803,4.500
3,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+3,2026-02-28,BCE1,D231,718589,5.662848,1.800,5.662848,4.500
4,ECOM,Hair Oils,ADV-AHO-R,B,2025-11-30,M+4,2026-03-31,BCE1,D231,718589,0.000000,2.700,4.500000,4.500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
282199,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+1,2025-11-30,QCW2,D461,718836,3.767871,4.908,5.040000,5.040
282200,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+2,2025-12-31,QCW2,D461,718836,4.907447,4.932,5.040000,5.040
282201,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+3,2026-01-31,QCW2,D461,718836,4.751205,4.260,5.040000,5.040
282202,QCOM,Male Grooming,SW_HR_WAX,C,2025-10-31,M+4,2026-02-28,QCW2,D461,718836,8.038001,4.776,8.038001,5.040


In [123]:
# delete_query = """
# DELETE FROM TRN_MIL_DF_SHARED
# WHERE "run_month" IN ('2025-12-31', '2026-01-31')
# """

# cur = dev_conn.cursor()
# cur.execute(delete_query)
# cur.close()


In [124]:
data_query = """
SELECT *
FROM TRN_MIL_DF_SHARED
   
"""

df = pd.read_sql(data_query, dev_conn)

In [125]:
df.columns = df.columns.str.lower()
df['run_month'] = pd.to_datetime(df['run_month'])
df['month'] = pd.to_datetime(df['month'])
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum)
0,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+1,2026-06-30,AURG,D3A4,718288,4.124703
1,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+2,2026-07-31,AURG,D3A4,718288,5.140667
2,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+3,2026-08-31,AURG,D3A4,718288,3.992805
3,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+4,2026-09-30,AURG,D3A4,718288,4.498084
4,GT,CNO,PCNO(R),A,2026-05-31,M+1,2026-06-30,AURG,D3A4,718299,29.470186
...,...,...,...,...,...,...,...,...,...,...,...
727933,GT,Foods,SF_SOYACN,B,2026-04-30,M+4,2026-08-31,WUP,D113,808562,0.076000
727934,GT,Foods,SAF_HONEY,C,2026-04-30,M+1,2026-05-31,WUP,D113,808712,0.049667
727935,GT,Foods,SAF_HONEY,C,2026-04-30,M+2,2026-06-30,WUP,D113,808712,0.049667
727936,GT,Foods,SAF_HONEY,C,2026-04-30,M+3,2026-07-31,WUP,D113,808712,0.049667


In [126]:
df[(df['month'] == '2026-04-30') & (df['channel']=='GT') & (df['run_month']=='2026-03-31')]['pred vol (roum)'].sum()


5346018.0036901245

In [127]:
# # make sure run_month is datetime
# df["run_month"] = pd.to_datetime(df["run_month"])
# upload_df["run_month"] = pd.to_datetime(upload_df["run_month"])

# # filter condition
# mask = (df["channel"] == "GT") & (df["run_month"] == "2026-01-31")

# # remove those rows
# df_filtered = df.loc[~mask]

# # concat upload_df
# final_df = pd.concat([df_filtered, upload_df], ignore_index=True)


In [128]:
df_heuristics.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol'],
      dtype='object')

In [129]:
df = df.merge(df_heuristics[['channel','asm', 'depot', 'psku', 'run_month', 'month','prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']], on = ['channel','asm', 'depot', 'psku', 'run_month', 'month'],
       how = 'left')

In [130]:
df.isnull().sum()

channel                       0
portfolio                     0
brand                         0
brand class                 422
run_month                     0
m month                       0
month                         0
asm                           0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol              522159
rf_vol                   522159
prophet heuristic vol    485879
rf heuristic vol         485879
dtype: int64

In [131]:
# df = df[df['Month'] == '2025-12-01']
# df

In [132]:
import numpy as np

df['final_channel'] = np.where(
    df['channel'].isin(['QCOM', 'GT']),
    df['channel'],
    np.where(
        df['channel'].isin(['ECOM', 'MT']) &
        df['asm'].astype(str).str.startswith('B'),
        'B2B',
        df['channel']
    )
)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,final_channel
0,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+1,2026-06-30,AURG,D3A4,718288,4.124703,NaN,NaN,NaN,NaN,GT
1,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+2,2026-07-31,AURG,D3A4,718288,5.140667,NaN,NaN,NaN,NaN,GT
2,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+3,2026-08-31,AURG,D3A4,718288,3.992805,NaN,NaN,NaN,NaN,GT
3,GT,Saffola Oils,SAFF GOLD,A,2026-05-31,M+4,2026-09-30,AURG,D3A4,718288,4.498084,NaN,NaN,NaN,NaN,GT
4,GT,CNO,PCNO(R),A,2026-05-31,M+1,2026-06-30,AURG,D3A4,718299,29.470186,NaN,NaN,NaN,NaN,GT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
727933,GT,Foods,SF_SOYACN,B,2026-04-30,M+4,2026-08-31,WUP,D113,808562,0.076000,NaN,NaN,NaN,NaN,GT
727934,GT,Foods,SAF_HONEY,C,2026-04-30,M+1,2026-05-31,WUP,D113,808712,0.049667,NaN,NaN,NaN,NaN,GT
727935,GT,Foods,SAF_HONEY,C,2026-04-30,M+2,2026-06-30,WUP,D113,808712,0.049667,NaN,NaN,NaN,NaN,GT
727936,GT,Foods,SAF_HONEY,C,2026-04-30,M+3,2026-07-31,WUP,D113,808712,0.049667,NaN,NaN,NaN,NaN,GT


In [133]:
df.columns

Index(['channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku', 'pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol', 'final_channel'],
      dtype='object')

In [134]:
df = df.groupby(['final_channel', 'portfolio', 'brand', 'brand class', 'run_month', 'm month',
       'month', 'asm', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()#['Channel'].unique()
df.rename(columns = {'final_channel':'channel'}, inplace = True)
df

,channel,portfolio,brand,brand class,run_month,m month,month,asm,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734910,0.0,0.000000,0.000000,0.0,0.0
1,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734988,0.0,0.000000,0.000000,0.0,0.0
2,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734993,0.0,0.000000,0.000000,0.0,0.0
3,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,734995,0.0,0.000000,0.000000,0.0,0.0
4,B2B,0,JH_FRG_L,0x2a,2026-05-31,M+1,2026-06-30,BCN2,D117,735043,0.0,0.000000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
716584,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809807,0.0,14.685752,2.232000,0.0,0.0
716585,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809850,0.0,0.000000,0.120000,0.0,0.0
716586,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809865,0.0,0.000000,0.746667,0.0,0.0
716587,QCOM,Skin Care,PURSNS_ML,C,2025-11-30,M+4,2026-03-31,QCW2,D461,809866,0.0,0.000000,0.000000,0.0,0.0


In [135]:
df[(df['run_month'] == '2026-04-30')]['channel'].unique()

array(['GT', 'MT'], dtype=object)

### Actuals

In [136]:
actuals_query = """
SELECT
    CM.channel_name,
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2023-01-31' AND '2026-07-31' AND
    CM.channel_name IN ('MT', 'E-Commerce', 'Q-Commerce', 'GT')
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY 1, 2, 3, 4, 6
"""

actuals_df = pd.read_sql(
    actuals_query,
    prod_conn
)

In [137]:
actuals_df.columns = actuals_df.columns.str.lower()
actuals_df['month_date'] = pd.to_datetime(actuals_df['month_date'])

In [138]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,E-Commerce,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000,0.808,0.076
1,E-Commerce,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525,0.774,0.229
2,E-Commerce,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549,0.724,0.364
3,E-Commerce,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189,0.282,0.057
4,E-Commerce,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335,0.528,-0.021


In [139]:
actuals_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [140]:
# actuals_df['key'] =  actuals_df['asm_area_code'].astype(str) + '_' + actuals_df['depot_code'].astype(str) + '_' + actuals_df['parent_material_code'].astype(str) 
# actuals_df

In [141]:
# actuals_df[(actuals_df['channel_name'] == 'GT') & (actuals_df['month_date']=='2026-02-28')]['key'].nunique()

In [142]:
actuals_df['channel_name'] = actuals_df['channel_name'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})

In [143]:
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.808,0.076
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.774,0.229
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.724,0.364
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.282,0.057
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.528,-0.021
...,...,...,...,...,...,...,...,...,...,...
2390165,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.000,0.062444,0.000,0.000
2390166,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-07-31,0.000,0.068349,0.000,0.000
2390167,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.000,13.636364,0.000,0.000
2390168,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.000,21.106628,0.000,0.000


In [144]:
import numpy as np

actuals_df['final_channel'] = np.where(
    actuals_df['channel_name'].isin(['QCOM', 'GT']),
    actuals_df['channel_name'],
    np.where(
        actuals_df['channel_name'].isin(['ECOM', 'MT']) &
        actuals_df['asm_area_code'].astype(str).str.startswith('B'),
        'B2B',
        actuals_df['channel_name']
    )
)
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,final_channel
0,ECOM,BCE1,D231,718297,PCNO(R),2023-01-31,0.000,0.000000,0.808,0.076,B2B
1,ECOM,BCE1,D231,718297,PCNO(R),2023-02-28,0.229,0.525000,0.774,0.229,B2B
2,ECOM,BCE1,D231,718297,PCNO(R),2023-03-31,0.364,0.549000,0.724,0.364,B2B
3,ECOM,BCE1,D231,718297,PCNO(R),2023-04-30,0.057,0.189000,0.282,0.057,B2B
4,ECOM,BCE1,D231,718297,PCNO(R),2023-05-31,-0.021,0.335000,0.528,-0.021,B2B
...,...,...,...,...,...,...,...,...,...,...,...
2390165,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.000,0.062444,0.000,0.000,QCOM
2390166,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-07-31,0.000,0.068349,0.000,0.000,QCOM
2390167,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.000,13.636364,0.000,0.000,QCOM
2390168,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.000,21.106628,0.000,0.000,QCOM


In [145]:
actuals_df = actuals_df.groupby(['final_channel', 'asm_area_code', 'depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,final_channel,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,PADV-HRCR,2024-09-30,0.0,0.000000,0.0,0.0
1,B2B,BCE1,D231,702478,PADV-HRCR,2024-10-31,0.0,0.000000,0.0,0.0
2,B2B,BCE1,D231,702478,PADV-HRCR,2024-12-31,0.0,0.000000,0.0,0.0
3,B2B,BCE1,D231,705148,NHR-UTTAM,2024-09-30,0.0,0.000000,0.0,0.0
4,B2B,BCE1,D231,705148,NHR-UTTAM,2024-10-31,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
2314457,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-06-30,0.0,0.062444,0.0,0.0
2314458,QCOM,QCW2,D463,811279,SAF_CDPRS,2026-07-31,0.0,0.068349,0.0,0.0
2314459,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-04-30,0.0,13.636364,0.0,0.0
2314460,QCOM,QCW2,D463,811287,PA_RSW_SR,2026-05-31,0.0,21.106628,0.0,0.0


In [146]:
vol_cols = [
    'pri_actuals_vol_rum',
    'pri_apo_plan_vol_rum',
    'sec_apo_plan_vol_rum',
    'sec_actuals_vol_rum'
]

actuals_df[vol_cols] = actuals_df[vol_cols].clip(lower=0)

In [147]:
actuals_df.rename(columns = {'final_channel':'channel_name'}, inplace = True)

In [148]:
tmp_df = pd.DataFrame()

for c in ['GT', 'MT', 'ECOM', 'QCOM','B2B']:
    tmp2_df = actuals_df[actuals_df['channel_name'] == c]
    tmp2_df = demand_driver_realign_pskus(tmp2_df, channel=c)
    tmp_df = pd.concat([tmp_df, tmp2_df], ignore_index=True)

In [149]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

26937

In [150]:
tmp_df['month_date'].unique()

<DatetimeArray>
['2023-02-28 00:00:00', '2023-03-31 00:00:00', '2023-04-30 00:00:00',
 '2023-05-31 00:00:00', '2023-06-30 00:00:00', '2023-07-31 00:00:00',
 '2023-08-31 00:00:00', '2023-09-30 00:00:00', '2023-10-31 00:00:00',
 '2023-11-30 00:00:00', '2023-12-31 00:00:00', '2024-01-31 00:00:00',
 '2024-02-29 00:00:00', '2024-03-31 00:00:00', '2024-04-30 00:00:00',
 '2024-05-31 00:00:00', '2024-06-30 00:00:00', '2024-07-31 00:00:00',
 '2024-08-31 00:00:00', '2024-09-30 00:00:00', '2024-10-31 00:00:00',
 '2024-11-30 00:00:00', '2024-12-31 00:00:00', '2025-01-31 00:00:00',
 '2025-02-28 00:00:00', '2025-03-31 00:00:00', '2025-04-30 00:00:00',
 '2025-05-31 00:00:00', '2025-06-30 00:00:00', '2025-07-31 00:00:00',
 '2025-08-31 00:00:00', '2025-09-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00', '2026-01-31 00:00:00', '2026-02-28 00:00:00',
 '2023-01-31 00:00:00', '2025-12-31 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00',
 '20

In [151]:
tmp_df = tmp_df.groupby(
    ['channel_name', 'asm_area_code', 'depot_code', 
     'parent_material_code', 'month_date'], as_index=False
).sum()

In [152]:
tmp_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [153]:
actuals_df = tmp_df.copy()

del tmp_df, tmp2_df

In [154]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,month_date,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,702478,2024-09-30,PADV-HRCR,0.0,0.0,0.0,0.0
1,B2B,BCE1,D231,702478,2024-10-31,PADV-HRCR,0.0,0.0,0.0,0.0
2,B2B,BCE1,D231,702478,2024-12-31,PADV-HRCR,0.0,0.0,0.0,0.0
3,B2B,BCE1,D231,705148,2024-09-30,NHR-UTTAM,0.0,0.0,0.0,0.0
4,B2B,BCE1,D231,705148,2024-10-31,NHR-UTTAM,0.0,0.0,0.0,0.0


In [155]:
actuals_df['sec_actuals_vol_rum'].sum()

236643667.96500006

In [156]:
# actuals_df = actuals_df.groupby(['channel_name', 'depot_code', 'parent_material_code',
#        'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
#        'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
# actuals_df

In [157]:
# actuals_df[actuals_df['channel_name'] == 'GT'].to_csv('trend_GT.csv')

In [158]:
actuals_df = actuals_df.rename(columns={
    'channel_name': 'channel',
    'asm_area_code': 'asm',
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol'
})

In [159]:
actuals_df.columns

Index(['channel', 'asm', 'depot', 'psku', 'month_date', 'material_group_code',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol'],
      dtype='object')

In [160]:
actuals_df = actuals_df.groupby(['channel', 'depot', 'psku', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'Consensus Vol',
       'Actuals Vol']].sum().reset_index()

In [161]:
df = df.groupby(['channel', 'portfolio', 'brand', 'run_month', 'm month',
       'month', 'depot', 'psku'])[['pred vol (roum)', 'prophet vol',
       'rf_vol', 'prophet heuristic vol', 'rf heuristic vol']].sum().reset_index()

In [162]:
df['psku'] = df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [163]:
actuals_df.rename(columns = {'month_date':'month'}, inplace = True)

In [164]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
533075,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0
533076,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0
533077,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0
533078,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0


In [165]:
# duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# # Get indices of duplicates that are NOT PABABY_ML
# indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# # Remove those rows
# actuals_df = actuals_df.drop(indices_to_drop)

# # Verify no duplicates remain
# print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

In [166]:
actuals_df[actuals_df.duplicated(subset=['channel', 'depot', 'psku', 'month'], keep=False)]

,channel,depot,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol


In [167]:
len_before_merge = len(df)
df = df.merge(
    actuals_df.drop([ 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel', 'depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [168]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533075,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
533076,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0,NaN,NaN
533077,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0,NaN,NaN
533078,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0,NaN,NaN


In [169]:
df[(df['channel'] == 'GT') & (df['month'] == '2026-05-31') & (df['run_month'] == '2026-04-30')]['pred vol (roum)'].sum()

5198880.692278133

In [170]:
df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna())]#['pred vol (roum)'].sum()#.isnull().sum()

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol
134508,GT,0,PA_SHMP_R,2026-05-31,M+3,2026-08-31,D112,732903,56.100000,0.0,0.0,0.0,0.0,NaN,NaN
134509,GT,0,PA_SHMP_R,2026-05-31,M+3,2026-08-31,D112,732908,29.070000,0.0,0.0,0.0,0.0,NaN,NaN
134510,GT,0,PA_SHMP_R,2026-05-31,M+3,2026-08-31,D112,732925,72.960000,0.0,0.0,0.0,0.0,NaN,NaN
134511,GT,0,PA_SHMP_R,2026-05-31,M+3,2026-08-31,D112,732929,30.600000,0.0,0.0,0.0,0.0,NaN,NaN
134512,GT,0,PA_SHMP_R,2026-05-31,M+3,2026-08-31,D112,732935,29.760000,0.0,0.0,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
316685,GT,Skin Care,PAD_BDYOL,2026-06-30,M+4,2026-10-31,D232,719118,0.000000,0.0,0.0,0.0,0.0,NaN,NaN
316686,GT,Skin Care,PAD_BDYOL,2026-06-30,M+4,2026-10-31,D236,719117,0.182498,0.0,0.0,0.0,0.0,NaN,NaN
316687,GT,Skin Care,PAD_BDYOL,2026-06-30,M+4,2026-10-31,D236,719118,2.919966,0.0,0.0,0.0,0.0,NaN,NaN
316688,GT,Skin Care,PAD_BDYOL,2026-06-30,M+4,2026-10-31,D463,719116,0.000000,0.0,0.0,0.0,0.0,NaN,NaN


In [171]:
# actuals_df[(actuals_df['asm'] == 'KARN') & (actuals_df['psku'] == 718314)]
actuals_df['key'] =  actuals_df['depot'] + actuals_df['psku'].astype(str)
df['key'] = df['depot'] + df['psku'].astype(str)


In [172]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [173]:
qtr_ind_rate_df[qtr_ind_rate_df['brand_code'] == 'PCNO(R)']

,month_date,brand_code,qtr_ind_rate
73,2027-03-31,PCNO(R),349274.00142


In [174]:
len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [175]:
df['brand'].unique()

array(['JH_FRG_L', 'PA_SHMP_R', 'PA_SHMP_S', 'NHR-UTTAM', 'PCNO FLEX',
       'PCNO(R)', 'CO_SO_VCN', 'SAF-MUSLI', 'SAFF OATS', 'SAFF SALT',
       'SAFF_ODLS', 'SAF_HONEY', 'SAF_MAYO', 'SAF_MILET', 'SAF_PNBTR',
       'SFOAT-CUP', 'SFOATS-FL', 'SFOATS_MG', 'SF_IM_CHY', 'SF_MNCHPS',
       'SF_SOYACN', 'ADV-AHO-R', 'H&C', 'H&C_ALMND', 'NHR NSJ H',
       'NHR-SABDM', 'NHRN_ALMD', 'NHR_ALOAM', 'NHR_SSAHO', 'NIHAR NHO',
       'PA-ALO-HO', 'PADV-HOT', 'PADVJAS-R', 'PADV_AMRO', 'PADV_SMPN',
       'PA_AMVITE', 'PA_CN_HO', 'PA_EXT_ML', 'PA_JASGLD', 'P_AL_GOLD',
       'P_EN_ALM', 'P_EN_BGHB', 'P_EN_CRSH', 'P_EN_RSMR', 'JH_SCR_KG',
       'JH_SCR_L', 'BRD_BDOIL', 'BRD_BDSPR', 'BRD_DOGAS', 'BRD_FSWSH',
       'BRD_HROIL', 'BRD_HRWAX', 'BRD_PERFM', 'PADV-HRCR', 'SW HRGEL',
       'SW HSPRY', 'SW NOGAS', 'SW STLDEO', 'SW_HR_WAX', 'NHR_VTEHO',
       'LVN_SHMP', 'MALO-NATU', 'MALT-NATU', 'REV.LQDST', 'REV.ST.',
       'REV_LQFRG', 'HC SNS', 'LIVON', 'LIVON S-R', 'LVN_SR_DR',
       'SAFF ACTV',

In [176]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,pred vol (roum),prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,Consensus Vol,Actuals Vol,key,Index Rate
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734910,NaN
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734988,NaN
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734993,NaN
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117734995,NaN
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D117735043,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
533075,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810406,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D674810406,1779.273746
533076,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D674,810407,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D674810407,1779.273746
533077,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,808489,0.0,9.892327,4.895,0.0,0.0,NaN,NaN,D676808489,1779.273746
533078,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+4,2026-03-31,D676,809750,0.0,0.000000,0.000,0.0,0.0,NaN,NaN,D676809750,1779.273746


In [177]:
df.isna().sum()

channel                       0
portfolio                     0
brand                         0
run_month                     0
m month                       0
month                         0
depot                         0
psku                          0
pred vol (roum)               0
prophet vol                   0
rf_vol                        0
prophet heuristic vol         0
rf heuristic vol              0
Consensus Vol            166716
Actuals Vol              166716
key                           0
Index Rate                  124
dtype: int64

In [178]:
df['Consensus Vol'] = df['Consensus Vol'].fillna(0)
df['Actuals Vol'] = df['Actuals Vol'].fillna(0)

In [179]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'pred vol (roum)', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'key', 'Index Rate'],
      dtype='object')

In [180]:
df.rename(columns={'pred vol (roum)': 'Stat Vol'}, inplace=True)

In [181]:
df['Stat Val'] = df['Stat Vol'] * df['Index Rate'] / (10 ** 7)
df['Consensus Val'] = df['Consensus Vol'] * df['Index Rate'] / (10 ** 7)
df['Actuals Val'] = df['Actuals Vol'] * df['Index Rate'] / (10 ** 7)

In [182]:
# df[(df['channel'] == 'GT') & (df['Actuals Vol'].isna()) & (df['run_month'] == '2026-03-31') & (df['m month'] == 'M+1')]['Stat Val'].sum()

In [239]:
df[(df['run_month'] == '2026-06-30') & (df['m month'] == 'M+1') & (df['channel'] == 'GT')]['Stat Val'].sum()

419.73863614950733

In [184]:
df['Stat Error'] = df['Stat Val'] - df['Actuals Val']
df['Consensus Error'] = df['Consensus Val'] - df['Actuals Val']

df['Stat Abs Error'] = np.abs(df['Stat Error'])
df['Consensus Abs Error'] = np.abs(df['Consensus Error'])

In [185]:
df.columns

Index(['channel', 'portfolio', 'brand', 'run_month', 'm month', 'month',
       'depot', 'psku', 'Stat Vol', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'Consensus Vol',
       'Actuals Vol', 'key', 'Index Rate', 'Stat Val', 'Consensus Val',
       'Actuals Val', 'Stat Error', 'Consensus Error', 'Stat Abs Error',
       'Consensus Abs Error'],
      dtype='object')

In [186]:
for col in ['prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol']:
    df[f'{col}_value'] = df[col] * df['Index Rate'] / (10 ** 7)

In [187]:
df['stat_bias'] = df['Stat Error']/df['Actuals Val']

In [188]:
df = df.fillna(0)

In [189]:
df['run_month'].unique()

<DatetimeArray>
['2026-05-31 00:00:00', '2026-06-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00', '2026-03-31 00:00:00', '2026-04-30 00:00:00']
Length: 9, dtype: datetime64[ns]

In [190]:
df = df[df['m month'] == 'M+1']#.isnull().sum()

In [191]:
import numpy as np
import pandas as pd

df['stat_bias'] = (
    df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df['stat_bias_bucket'] = pd.cut(
    df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False   # e
)


In [192]:
df

,channel,portfolio,brand,run_month,m month,month,depot,psku,Stat Vol,prophet vol,...,Stat Error,Consensus Error,Stat Abs Error,Consensus Abs Error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
1,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
2,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
3,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
4,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532661,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810406,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
532662,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D674,810407,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%
532663,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,808489,0.0,23.308381,...,0.0,0.0,0.0,0.0,0.004147,0.001373,0.0,0.0,0.0,0% to 5%
532664,QCOM,Skin Care,PURSNS_ML,2025-11-30,M+1,2025-12-31,D676,809750,0.0,0.000000,...,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0% to 5%


In [193]:
df['forecast_granularity'] = 'Depot x PSKU'
df['forecast_type'] = 'secondary'

In [194]:
cols = ['forecast_granularity', 'forecast_type'] + [
    c for c in df.columns
    if c not in ['forecast_granularity', 'forecast_type']
]

df = df[cols]

In [195]:
df['run_month'].unique()

<DatetimeArray>
['2026-05-31 00:00:00', '2026-06-30 00:00:00', '2025-10-31 00:00:00',
 '2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00', '2026-03-31 00:00:00', '2026-04-30 00:00:00']
Length: 9, dtype: datetime64[ns]

### Qcom

### Helper Functions

In [196]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

### Forecast

In [240]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_SHARED""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [198]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,depot,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-11-30,0x2a,718287,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
1,2025-11-30,0x2a,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,Qcom
2,2025-11-30,0x2a,718297,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
3,2025-11-30,0x2a,718299,PCNO(R),CNO,M,2025-11-30,0.0,Qcom
4,2025-11-30,0x2a,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,Qcom
...,...,...,...,...,...,...,...,...,...
680487,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680488,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680489,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680490,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM


### Actuals

In [241]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [242]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2023-01-01' AND '2026-07-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [243]:
depot_psku_primary_df['channel'] = 'QCOM'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [202]:
# depot_psku_primary_df_qcom.to_csv('Trend_qcom.csv')

In [244]:
depot_psku_primary_query = """
SELECT
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-01' AND '2026-07-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['depot_code', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['depot_code', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [245]:
depot_psku_primary_df['channel'] = 'ECOM'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [246]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,D112,715098,CO_SO_PCP,2023-07-31,0.0,0.000000,1.762,0.0,QCOM
1,D112,715098,CO_SO_PCP,2023-08-31,0.0,0.000000,1.762,0.0,QCOM
2,D112,715099,CO_SO_PCP,2023-07-31,0.0,0.000000,1.762,0.0,QCOM
3,D112,715099,CO_SO_PCP,2023-08-31,0.0,0.000000,1.762,0.0,QCOM
4,D112,715100,CO_SO_PCP,2023-02-28,0.0,0.000000,0.284,0.0,QCOM
...,...,...,...,...,...,...,...,...,...
190958,D677,811287,PA_RSW_SR,2026-05-31,0.0,10.794646,0.000,0.0,ECOM
190959,D677,811287,PA_RSW_SR,2026-06-30,0.0,12.332112,0.000,0.0,ECOM
190960,D677,811287,PA_RSW_SR,2026-07-31,0.0,9.672414,0.000,0.0,ECOM
190961,D677,811416,SAF-MUSLI,2026-06-30,0.0,0.121222,0.000,0.0,ECOM


In [247]:
actuals_df = depot_psku_primary_df.copy()

In [248]:
actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum()

451

In [249]:
# For duplicates, keep only the row with PABABY_ML material_group_code
duplicates = actuals_df[actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date'], keep=False)]

# Get indices of duplicates that are NOT PABABY_ML
indices_to_drop = duplicates[duplicates['material_group_code'] != 'PABABY_ML'].index

# Remove those rows
actuals_df = actuals_df.drop(indices_to_drop)

# Verify no duplicates remain
print(actuals_df.duplicated(subset=['channel','depot_code', 'parent_material_code','month_date']).sum())

0


In [250]:
actuals_df = actuals_df.groupby(['channel','depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,ECOM,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,ECOM,D112,718288,SAFF GOLD,2026-05-31,0.000,0.028975,0.0000,0.000
3,ECOM,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
4,ECOM,D112,718299,PCNO(R),2026-05-31,0.057,0.074845,0.0557,0.057
...,...,...,...,...,...,...,...,...,...
190193,QCOM,D677,811279,SAF_CDPRS,2026-06-30,0.000,0.045348,0.0000,0.000
190194,QCOM,D677,811279,SAF_CDPRS,2026-07-31,0.000,0.072285,0.0000,0.000
190195,QCOM,D677,811287,PA_RSW_SR,2026-04-30,0.000,8.823530,0.0000,0.000
190196,QCOM,D677,811287,PA_RSW_SR,2026-05-31,0.000,14.774640,0.0000,0.000


In [251]:
actuals_df.columns

Index(['channel', 'depot_code', 'parent_material_code', 'material_group_code',
       'month_date', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum',
       'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum'],
      dtype='object')

In [252]:
actuals_df = actuals_df.rename(columns={
    'depot_code': 'depot',
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [253]:
actuals_df.head()

,channel,depot,psku,material_group_code,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,ECOM,D112,718287,PCNO(R),2025-12-31,0.000,0.023463,0.0000,0.000
1,ECOM,D112,718288,SAFF GOLD,2026-03-31,0.000,0.000000,1.9086,0.000
2,ECOM,D112,718288,SAFF GOLD,2026-05-31,0.000,0.028975,0.0000,0.000
3,ECOM,D112,718299,PCNO(R),2026-04-30,0.000,0.038390,0.0618,0.000
4,ECOM,D112,718299,PCNO(R),2026-05-31,0.057,0.074845,0.0557,0.057


In [213]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [254]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [215]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [255]:
qcom_df['run_month'].unique()

<DatetimeArray>
['2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00', '2026-03-31 00:00:00', '2026-04-30 00:00:00',
 '2026-05-31 00:00:00', '2026-06-30 00:00:00']
Length: 8, dtype: datetime64[ns]

In [257]:
qcom_df['channel'] = qcom_df['channel'].str.upper()
qcom_df['depot'] = qcom_df['depot'].str.upper()

qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-11-30,0X2A,718287,PCNO(R),CNO,M,2025-11-30,0.0,QCOM
1,2025-11-30,0X2A,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,QCOM
2,2025-11-30,0X2A,718297,PCNO(R),CNO,M,2025-11-30,0.0,QCOM
3,2025-11-30,0X2A,718299,PCNO(R),CNO,M,2025-11-30,0.0,QCOM
4,2025-11-30,0X2A,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,QCOM
...,...,...,...,...,...,...,...,...,...
680487,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680488,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680489,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM
680490,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM


In [258]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([ 'material_group_code', 'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','depot', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [259]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [260]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [261]:
qcom_df.isna().sum()

month                          0
depot                          0
psku                           0
brand                          0
portfolio                      0
m month                        0
run_month                      0
calculated primary vol         0
channel                        0
Consensus Vol             508723
Actuals Vol               508723
Index Rate                     0
dtype: int64

In [262]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [263]:
qcom_df['Consensus Vol'].max()

21077.84

In [264]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-11-30,0X2A,718287,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,0.0,349274.001420
1,2025-11-30,0X2A,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,QCOM,0.0,0.0,138865.260689
2,2025-11-30,0X2A,718297,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,0.0,349274.001420
3,2025-11-30,0X2A,718299,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,0.0,349274.001420
4,2025-11-30,0X2A,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,QCOM,0.0,0.0,230000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
680487,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,0.0,12860.631072
680488,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,0.0,12860.631072
680489,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,0.0,12860.631072
680490,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM,0.0,0.0,260000.000000


In [265]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [266]:
qcom_df['channel'].unique()

array(['QCOM', 'ECOM'], dtype=object)

In [271]:
qcom_df[(qcom_df['month'] == '2026-07-31') & (qcom_df['channel']=='ECOM') & (qcom_df['run_month']=='2026-06-30')]['Actuals Val'].sum()

42.43565860892847

In [273]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [274]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [275]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2025-11-30,0X2A,718287,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,2025-11-30,0X2A,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,QCOM,0.0,...,138865.260689,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,2025-11-30,0X2A,718297,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,2025-11-30,0X2A,718299,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,349274.001420,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,2025-11-30,0X2A,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,QCOM,0.0,...,230000.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680487,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,12860.631072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
680488,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,12860.631072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
680489,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,12860.631072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
680490,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,260000.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%


In [ ]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [276]:
df.columns = df.columns.str.lower()

In [277]:
qcom_df.columns

Index(['month', 'depot', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [278]:

qcom_df['forecast_granularity'] = 'Depot x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,depot,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2025-11-30,0X2A,718287,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
1,2025-11-30,0X2A,718288,SAFF GOLD,Saffola Oils,M,2025-11-30,0.0,QCOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
2,2025-11-30,0X2A,718297,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
3,2025-11-30,0X2A,718299,PCNO(R),CNO,M,2025-11-30,0.0,QCOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
4,2025-11-30,0X2A,718300,PCNO FLEX,CNO,M,2025-11-30,0.0,QCOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680487,2026-10-31,D677,811267,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
680488,2026-10-31,D677,811268,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
680489,2026-10-31,D677,811269,PA_ESS_HO,Hair Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary
680490,2026-10-31,D677,811279,SAF_CDPRS,Saffola Oils,M+4,2026-06-30,0.0,ECOM,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%,Depot x PSKU,offtakes_to_primary


In [279]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [280]:
df[(df['channel'] == 'MT') & (df['run_month'] == '2026-06-30') & (df['month'] == '2026-07-31')]['stat val'].sum()

121.04096548610876

In [281]:
final_df = pd.concat([df,qcom_df])
final_df

,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,psku,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624591,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811267,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624592,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811268,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624593,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811269,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624594,Depot x PSKU,offtakes_to_primary,ECOM,Saffola Oils,SAF_CDPRS,2026-06-30,M+1,2026-07-31,D677,811279,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%


### offtakes

In [283]:
query = """select * from TRN_MIL_DF_OFFTAKES_OUTPUT"""
df_offtakes = pd.read_sql(query, dev_conn)
df_offtakes.columns = df_offtakes.columns.str.lower()
df_offtakes['run_month'] = pd.to_datetime(df_offtakes['run_month'])
df_offtakes['month_date'] = pd.to_datetime(df_offtakes['month_date'])
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
0,2026-01-31,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M,60.461823,58.398089,0.830145,QCOM
1,2026-01-31,Blinkit,718310,PCNO(R),CNO,2026-01-31,M,0.000000,0.000000,0.000790,QCOM
2,2026-01-31,Blinkit,718312,PCNO(R),CNO,2026-01-31,M,8.842138,6.024962,0.281155,QCOM
3,2026-01-31,Blinkit,718315,PCNO(R),CNO,2026-01-31,M,0.000000,0.000300,0.000000,QCOM
4,2026-01-31,Blinkit,718317,H&C,Hair Oils,2026-01-31,M,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
196030,2027-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.002317,ECOM
196031,2027-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,2026-06-30,M+9,0.000000,0.000000,0.000672,ECOM
196032,2027-03-31,Nykaa,810738,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000000,ECOM
196033,2027-03-31,Nykaa,810805,PABABY_GM,Skin Care,2026-06-30,M+9,0.000000,0.000000,0.000544,ECOM


In [284]:
df_offtakes['run_month'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00']
Length: 6, dtype: datetime64[ns]

In [285]:
query_qcom = """select * from TRN_DF_QCOM_OFFTAKE_CHAIN_DEPOT_PSKU
where run_month = '2026-08-31'
and month_date between '2026-01-31' and '2026-07-31'
"""
qcom_df = pd.read_sql(query_qcom, dev_conn)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month_date'] = pd.to_datetime(qcom_df['month_date'])
qcom_df['channel'] = 'QCOM'
qcom_df

,month_date,key,platform_name,depot,parent_material_code,brand_code,vol_in_rum,imputed,run_month,channel
0,2026-01-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-08-31,QCOM
1,2026-02-28,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-08-31,QCOM
2,2026-03-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-08-31,QCOM
3,2026-04-30,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-08-31,QCOM
4,2026-05-31,blinkit_D112_718288,blinkit,D112,718288,SAFF GOLD,0.0,1,2026-08-31,QCOM
...,...,...,...,...,...,...,...,...,...,...
73283,2026-03-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-08-31,QCOM
73284,2026-04-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-08-31,QCOM
73285,2026-05-31,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-08-31,QCOM
73286,2026-06-30,zepto_D677_810439,zepto,D677,810439,SAF-MUSLI,0.0,1,2026-08-31,QCOM


In [286]:
qcom_df.columns

Index(['month_date', 'key', 'platform_name', 'depot', 'parent_material_code',
       'brand_code', 'vol_in_rum', 'imputed', 'run_month', 'channel'],
      dtype='object')

In [287]:
qcom_df = qcom_df.groupby(['channel', 'run_month', 'month_date', 'platform_name', 'parent_material_code',
       'brand_code'])['vol_in_rum'].sum().reset_index()
qcom_df

,channel,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
0,QCOM,2026-08-31,2026-01-31,blinkit,718288,SAFF GOLD,62.478
1,QCOM,2026-08-31,2026-01-31,blinkit,718310,PCNO(R),0.001
2,QCOM,2026-08-31,2026-01-31,blinkit,718312,PCNO(R),5.240
3,QCOM,2026-08-31,2026-01-31,blinkit,718315,PCNO(R),0.000
4,QCOM,2026-08-31,2026-01-31,blinkit,718317,H&C,0.000
...,...,...,...,...,...,...,...
5886,QCOM,2026-08-31,2026-07-31,zepto,810971,PA_ESS_HO,2.070
5887,QCOM,2026-08-31,2026-07-31,zepto,811005,PA_ESS_HO,0.966
5888,QCOM,2026-08-31,2026-07-31,zepto,811019,PADV_WIPS,0.000
5889,QCOM,2026-08-31,2026-07-31,zepto,811021,PABABY_GM,17.014


In [288]:
query_ecom = """select * from TRN_DF_ECOM_OFFTAKE_CHAIN_PSKU_NEW
where run_month = '2026-07-31'
and month_date between '2026-01-31' and '2026-06-30'
"""
ecom_df = pd.read_sql(query_ecom, dev_conn)
ecom_df.columns = ecom_df.columns.str.lower()
ecom_df['run_month'] = pd.to_datetime(ecom_df['run_month'])
ecom_df['month_date'] = pd.to_datetime(ecom_df['month_date'])
ecom_df['channel'] = 'ECOM'
ecom_df

,month_date,platform_name,parent_material_code,vol_in_rum,indexbpm,brand_code,imputed,big_billion_days,big_billion_days_lag_1,big_billion_days_lag_2,...,big_billion_days_lead_2,great_indian_festival,great_indian_festival_lag_1,great_indian_festival_lag_2,great_indian_festival_lead_1,great_indian_festival_lead_2,ratio_last_year,quarter,run_month,channel
0,2026-01-31,Amazon ARIPL,718288,22.242,30.61899,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.837698,1,2026-07-31,ECOM
1,2026-02-28,Amazon ARIPL,718288,20.178,27.77763,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.670088,1,2026-07-31,ECOM
2,2026-03-31,Amazon ARIPL,718288,39.372,54.20065,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.850513,1,2026-07-31,ECOM
3,2026-04-30,Amazon ARIPL,718288,12.102,16.80547,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,0.831900,2,2026-07-31,ECOM
4,2026-05-31,Amazon ARIPL,718288,30.204,41.94286,SAFF GOLD,0,0,0,0,...,0,0,0,0,0,0,1.122864,2,2026-07-31,ECOM
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18646,2026-02-28,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0,0.000000,1,2026-07-31,ECOM
18647,2026-03-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0,NaN,1,2026-07-31,ECOM
18648,2026-04-30,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0,NaN,2,2026-07-31,ECOM
18649,2026-05-31,Nykaa,811021,0.000,0.00000,PADV_WIPS,1,0,0,0,...,0,0,0,0,0,0,NaN,2,2026-07-31,ECOM


In [289]:
ecom_df = ecom_df[['channel', 'run_month', 'month_date', 'platform_name',
       'parent_material_code', 'brand_code', 'vol_in_rum']]

In [290]:
offtake_actuals = pd.concat([qcom_df, ecom_df], ignore_index=True)
offtake_actuals

,channel,run_month,month_date,platform_name,parent_material_code,brand_code,vol_in_rum
0,QCOM,2026-08-31,2026-01-31,blinkit,718288,SAFF GOLD,62.478
1,QCOM,2026-08-31,2026-01-31,blinkit,718310,PCNO(R),0.001
2,QCOM,2026-08-31,2026-01-31,blinkit,718312,PCNO(R),5.240
3,QCOM,2026-08-31,2026-01-31,blinkit,718315,PCNO(R),0.000
4,QCOM,2026-08-31,2026-01-31,blinkit,718317,H&C,0.000
...,...,...,...,...,...,...,...
24537,ECOM,2026-07-31,2026-02-28,Nykaa,811021,PADV_WIPS,0.000
24538,ECOM,2026-07-31,2026-03-31,Nykaa,811021,PADV_WIPS,0.000
24539,ECOM,2026-07-31,2026-04-30,Nykaa,811021,PADV_WIPS,0.000
24540,ECOM,2026-07-31,2026-05-31,Nykaa,811021,PADV_WIPS,0.000


In [291]:
df_offtakes['run_month'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00',
 '2026-04-30 00:00:00', '2026-05-31 00:00:00', '2026-06-30 00:00:00']
Length: 6, dtype: datetime64[ns]

In [ ]:
# df_offtakes = df_offtakes[df_offtakes['run_month'].isin(['2026-02-28', '2026-01-31'])]
df_offtakes = df_offtakes[df_offtakes['m month'] == 'M+1']
#df_offtakes = df_offtakes[(df_offtakes['m month'] == 'M+1') & (df_offtakes['channel'] == 'QCOM')]

df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
837,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M+1,60.295965,57.833437,0.829005,QCOM
838,2026-02-28,Blinkit,718310,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000790,QCOM
839,2026-02-28,Blinkit,718312,PCNO(R),CNO,2026-01-31,M+1,8.137921,5.823517,0.281155,QCOM
840,2026-02-28,Blinkit,718315,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000000,QCOM
841,2026-02-28,Blinkit,718317,H&C,Hair Oils,2026-01-31,M+1,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
164568,2026-07-31,zepto,810685,SAF-MUSLI,Foods,2026-06-30,M+1,0.000000,0.000000,0.000021,QCOM
164569,2026-07-31,zepto,810738,PABABY_GM,Skin Care,2026-06-30,M+1,0.000000,0.000000,0.004203,QCOM
164570,2026-07-31,zepto,810971,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000984,QCOM
164571,2026-07-31,zepto,811005,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000738,QCOM


In [293]:
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
837,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M+1,60.295965,57.833437,0.829005,QCOM
838,2026-02-28,Blinkit,718310,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000790,QCOM
839,2026-02-28,Blinkit,718312,PCNO(R),CNO,2026-01-31,M+1,8.137921,5.823517,0.281155,QCOM
840,2026-02-28,Blinkit,718315,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000000,QCOM
841,2026-02-28,Blinkit,718317,H&C,Hair Oils,2026-01-31,M+1,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
164568,2026-07-31,zepto,810685,SAF-MUSLI,Foods,2026-06-30,M+1,0.000000,0.000000,0.000021,QCOM
164569,2026-07-31,zepto,810738,PABABY_GM,Skin Care,2026-06-30,M+1,0.000000,0.000000,0.004203,QCOM
164570,2026-07-31,zepto,810971,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000984,QCOM
164571,2026-07-31,zepto,811005,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000738,QCOM


In [294]:
offtake_actuals['channel'].unique()

array(['QCOM', 'ECOM'], dtype=object)

In [295]:
df_offtakes['channel'].unique()

array(['QCOM'], dtype=object)

In [296]:
df_offtakes

,month_date,platform_name,parent_material_code,brand_code,portfolio,run_month,m month,pred_prophet,pred_rf,final_heuristic_prophet_value_2,channel
837,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,2026-01-31,M+1,60.295965,57.833437,0.829005,QCOM
838,2026-02-28,Blinkit,718310,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000790,QCOM
839,2026-02-28,Blinkit,718312,PCNO(R),CNO,2026-01-31,M+1,8.137921,5.823517,0.281155,QCOM
840,2026-02-28,Blinkit,718315,PCNO(R),CNO,2026-01-31,M+1,0.000000,0.000000,0.000000,QCOM
841,2026-02-28,Blinkit,718317,H&C,Hair Oils,2026-01-31,M+1,0.000000,0.049697,0.000000,QCOM
...,...,...,...,...,...,...,...,...,...,...,...
164568,2026-07-31,zepto,810685,SAF-MUSLI,Foods,2026-06-30,M+1,0.000000,0.000000,0.000021,QCOM
164569,2026-07-31,zepto,810738,PABABY_GM,Skin Care,2026-06-30,M+1,0.000000,0.000000,0.004203,QCOM
164570,2026-07-31,zepto,810971,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000984,QCOM
164571,2026-07-31,zepto,811005,PA_ESS_HO,Hair Oils,2026-06-30,M+1,0.000000,0.000000,0.000738,QCOM


In [ ]:
df_offtakes['platform_name'] = df_offtakes['platform_name'].str.lower()
offtake_actuals['platform_name'] = offtake_actuals['platform_name'].str.lower()
df_offtakes['parent_material_code'] = df_offtakes['parent_material_code'].astype(int)
offtake_actuals['parent_material_code'] = offtake_actuals['parent_material_code'].astype(int)

In [ ]:
offtake_actuals[['channel', 'platform_name', 'parent_material_code','month_date', 'vol_in_rum']].dtypes

channel                         object
platform_name                   object
parent_material_code             int64
month_date              datetime64[ns]
vol_in_rum                     float64
dtype: object

In [ ]:
df_offtakes[['channel', 'platform_name', 'parent_material_code','month_date']].dtypes

channel                         object
platform_name                   object
parent_material_code             int64
month_date              datetime64[ns]
dtype: object

In [ ]:
x = df_offtakes.copy()

In [ ]:
# df_offtakes = x.copy()

In [ ]:
df_offtakes = df_offtakes.merge(
    offtake_actuals[['channel', 'platform_name', 'parent_material_code','month_date', 'vol_in_rum']]
    ,
    on=['channel', 'platform_name', 'parent_material_code','month_date'],
    how='left'
)

In [ ]:
df_offtakes.rename(columns={'brand_code':'brand'}, inplace=True)

In [ ]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()
len_before_merge = len(df_offtakes)
df_offtakes = df_offtakes.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df_offtakes)
del len_before_merge

In [ ]:
df_offtakes.rename(columns={'month_date':'month', 'final_heuristic_prophet_value_2':'offtakes_forecasted_value'}, inplace=True)
df_offtakes['offtakes_forecasted_vol'] = df_offtakes['offtakes_forecasted_value'] * (10 ** 7) / df_offtakes['Index Rate']
df_offtakes['offtakes actuals val'] = df_offtakes['vol_in_rum'] * df_offtakes['Index Rate'] / (10 ** 7)
df_offtakes['offtakes error'] = df_offtakes['offtakes actuals val'] - df_offtakes['offtakes_forecasted_value'] 
df_offtakes['offtakes abs error'] = np.abs(df_offtakes['offtakes error'])
df_offtakes


,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error
0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,ECOM,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594
1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000
2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,ECOM,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519
3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000
4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,ECOM,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20595,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000
20596,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,ECOM,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103
20597,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,ECOM,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232
20598,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,ECOM,NaN,366.484998,0.000000,NaN,NaN,NaN


In [ ]:
df_offtakes[df_offtakes['channel']=='QCOM'].groupby(['month'])['offtakes actuals val'].sum()

month
2026-02-28    28.104255
2026-03-31    33.739546
2026-04-30    27.589769
2026-05-31    30.593245
2026-06-30    29.511915
Name: offtakes actuals val, dtype: float64

In [ ]:
df_offtakes['offtakes_bias'] = df_offtakes['offtakes error']/df_offtakes['offtakes actuals val']
df_offtakes = df_offtakes.fillna(0)

df_offtakes['offtakes_bias'] = (
    df_offtakes['offtakes_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

df_offtakes['offtakes_bias_bucket'] = pd.cut(
    df_offtakes['offtakes_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [ ]:
df_offtakes['forecast_granularity'] = 'Chain x PSKU'
df_offtakes['forecast_type'] = 'offtakes'

In [ ]:
df_offtakes

,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,offtakes_forecasted_value,...,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,...,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,...,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,...,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,...,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,...,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20595,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,...,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
20596,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,...,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
20597,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,...,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
20598,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,...,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [ ]:
# df_offtakes.to_csv('acc_offtakes_till_may.csv')

In [ ]:
# df_stat_fva = pd.read_excel('/data/aman_singh/acuuracy_check/fva_and_stat_accuracy.xlsx', sheet_name = 'fva_depot_psku_final')
# df_stat_fva

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,dp error drm,dp abs error drm,dp error cam,dp abs error cam,dp error bam,dp abs error bam,Bias bucket drm,bias bucket cam,bias_bucket_bam,bias_bucket_plan
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142881,Depot x PSKU,offtakes_to_primary,QCOM,D677,810521,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012889,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142882,Depot x PSKU,offtakes_to_primary,QCOM,D677,810522,SAF_CDPRS,Saffola Oils,2026-03-31,M+1,0.012509,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142883,Depot x PSKU,offtakes_to_primary,QCOM,D677,810738,PABABY_GM,Skin Care,2026-03-31,M+1,11.106341,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%
142884,Depot x PSKU,offtakes_to_primary,QCOM,D677,810805,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,...,0.0,0.0,0.0,0.0,0.0,0.0,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%,5 | 0% to 5%


In [ ]:
# df_offtakes.rename(columns={'platform_name':'chain', 'parent_material_code':'psku'
#                             ,'Index Rate':'index rate'}, inplace=True)
# df_offtakes.drop(['vol_in_rum','pred_prophet', 'pred_rf','run_month'], axis=1, inplace=True)
# df_offtakes

,month,chain,psku,brand,portfolio,m month,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,2026-02-28,Blinkit,718288,SAFF GOLD,Saffola Oils,M+1,0.829005,QCOM,138865.260689,59.698539,0.841190,0.012185,0.012185,0.014485,0% to 5%,Chain x PSKU,offtakes
1,2026-02-28,Blinkit,718310,PCNO(R),CNO,M+1,0.000790,QCOM,349274.001420,0.022616,0.000017,-0.000772,0.000772,-44.231134,< -15%,Chain x PSKU,offtakes
2,2026-02-28,Blinkit,718312,PCNO(R),CNO,M+1,0.281155,QCOM,349274.001420,8.049687,0.194406,-0.086749,0.086749,-0.446225,< -15%,Chain x PSKU,offtakes
3,2026-02-28,Blinkit,718315,PCNO(R),CNO,M+1,0.000000,QCOM,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,2026-02-28,Blinkit,718317,H&C,Hair Oils,M+1,0.000000,QCOM,388.076436,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8297,2026-03-31,Nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
8298,2026-03-31,Nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.002456,ECOM,12860.631072,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%,Chain x PSKU,offtakes
8299,2026-03-31,Nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000933,ECOM,12860.631072,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%,Chain x PSKU,offtakes
8300,2026-03-31,Nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [ ]:
# final_df = pd.concat([df_stat_fva, df_offtakes], ignore_index=True)
# final_df

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,bias_bucket_bam,bias_bucket_plan,chain,offtakes_forecasted_value,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151183,Chain x PSKU,offtakes,ECOM,NaN,810605,KAYA_ML,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
151184,Chain x PSKU,offtakes,ECOM,NaN,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.002456,1.909829,0.002269,-0.000188,0.000188,-0.082669,-10% to -5%
151185,Chain x PSKU,offtakes,ECOM,NaN,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000933,0.725548,0.000522,-0.000411,0.000411,-0.787063,< -15%
151186,Chain x PSKU,offtakes,ECOM,NaN,810738,PABABY_GM,Skin Care,2026-03-31,M+1,NaN,...,NaN,NaN,Nykaa,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%


In [ ]:
# final_df.columns

Index(['forecast_granularity', 'forecast_type', 'channel', 'depot', 'psku',
       'brand', 'portfolio', 'month', 'm month', 'stat vol', 'actuals vol',
       'consensus vol', 'index rate', 'stat val', 'consensus val',
       'actuals val', 'stat error', 'consensus error', 'stat abs error',
       'consensus abs error', 'stat_bias', 'stat_bias_bucket',
       'delivery_vol_drm', 'delivery val_drm', 'delivery_vol_cam',
       'delivery val_cam', 'delivery_vol_bam', 'delivery val_bam',
       'brand class', 'dp error drm', 'dp abs error drm', 'dp error cam',
       'dp abs error cam', 'dp error bam', 'dp abs error bam',
       'Bias bucket drm', 'bias bucket cam', 'bias_bucket_bam',
       'bias_bucket_plan', 'chain', 'offtakes_forecasted_value',
       'offtakes_forecasted_vol', 'offtakes actuals val', 'offtakes error',
       'offtakes abs error', 'offtakes_bias', 'offtakes_bias_bucket'],
      dtype='object')

In [ ]:
# final_df.to_csv('/data/aman_singh/acuuracy_check/fva_stat_offtake_accuracy.csv', index=False)

### Chain psku primary accuracy

In [297]:
qcom_df = pd.read_sql(
    """select * from TRN_MIL_DF_OFT2PRIM_CPSKU""",
    dev_conn
)
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df['run_month'] = pd.to_datetime(qcom_df['run_month'])
qcom_df['month'] = pd.to_datetime(qcom_df['month'])

In [298]:
qcom_df#[qcom_df['Channel']=='Ecom']

,month,chain,psku,brand,portfolio,m month,run_month,calculated primary vol,channel
0,2025-12-31,Blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
1,2025-12-31,Blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,Qcom
2,2025-12-31,Blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
3,2025-12-31,Blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,Qcom
4,2025-12-31,Blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,Qcom
...,...,...,...,...,...,...,...,...,...
152443,2026-09-30,Zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,Qcom
152444,2026-09-30,Zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,Qcom
152445,2026-09-30,Zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom
152446,2026-09-30,Zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,Qcom


In [300]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

# realignment_df = realignment_df[
#     realignment_df['channel'].isin(['QCOM', 'QCOM B2C', 'ALL'])]


def realign_pskus(data, column, channel='QCOM'):
    realignment_data = realignment_df.copy()
    realignment_data = realignment_data[
        realignment_data['channel'].isin([channel, channel + ' B2C', 'ALL'])
    ]

    assert realignment_data['psku old'].dtype == 'int64'
    assert realignment_data['psku new'].dtype == 'int64'
    realignment_data = realignment_data[['psku old', 'psku new']].drop_duplicates()
    realignment_data = realignment_data.set_index('psku old').to_dict()['psku new']

    
    data[column] = data[column].astype(int)

    for old_psku, new_psku in realignment_data.items():
        data.loc[
            data[column] == old_psku, column
        ] = new_psku

    return data

In [301]:
chain_psku_primary_query = """
SELECT
    CASE
        WHEN MCM.chain = 'Kiranakart Technologies' THEN 'Zepto'
        WHEN MCM.chain = 'Grofers' THEN 'Blinkit'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Grofers', 'Zepto', 'Kiranakart Technologies', 'Swiggy', 'ZEPTO')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM on MESR.distributor_code = CM.customer_code
where month_date BETWEEN '2025-12-31' AND '2026-07-31' 
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""

depot_psku_primary_df = pd.read_sql(
    chain_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [302]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
1263,Blinkit,732296,PABABY_ML,2025-12-31,0.000,0.000000,55.5023,0.000
1264,Blinkit,732296,PABABY_ML,2026-01-31,0.000,161.650358,116.1056,0.000
1265,Blinkit,732296,PABABY_SP,2025-12-31,0.000,0.000000,0.0000,33.600
1266,Blinkit,732296,PABABY_SP,2026-01-31,0.000,0.000000,0.0000,0.000
1273,Blinkit,732297,PABABY_ML,2025-12-31,0.000,0.000000,100.1441,0.000
1274,Blinkit,732297,PABABY_ML,2026-01-31,0.000,397.753905,283.5170,0.000
1275,Blinkit,732297,PABABY_ML,2026-02-28,0.000,0.000000,0.0000,0.000
1276,Blinkit,732297,PABABY_ML,2026-06-30,9.840,0.000000,0.0000,9.840
1277,Blinkit,732297,PABABY_SP,2025-12-31,0.000,0.000000,0.0000,137.760
1278,Blinkit,732297,PABABY_SP,2026-01-31,58.220,0.000000,0.0000,58.220


In [303]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


10

In [304]:
depot_psku_primary_df[depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date'], keep=False)]

,chain,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
1725,Blinkit,811021,PABABY_GM,2026-04-30,52.128,0.000000,0.0000,52.128
1726,Blinkit,811021,PABABY_GM,2026-05-31,0.000,0.000000,0.0000,0.000
1727,Blinkit,811021,PABABY_GM,2026-07-31,69.504,0.000000,0.0000,69.504
1728,Blinkit,811021,PADV_WIPS,2026-04-30,217.200,215.037675,247.6281,0.000
1729,Blinkit,811021,PADV_WIPS,2026-05-31,286.704,421.887556,357.7630,286.704
1731,Blinkit,811021,PADV_WIPS,2026-07-31,0.000,437.094196,544.3660,0.000
2166,Swiggy,718647,LIVON S-R,2025-12-31,0.000,0.000000,277.5746,50.400
2168,Swiggy,718647,LIVON S-R,2026-02-28,122.400,370.404138,296.9990,122.400
2170,Swiggy,718647,LIVON S-R,2026-04-30,0.000,195.462769,195.9974,0.000
2174,Swiggy,718647,LVNPST_ML,2025-12-31,0.000,0.000000,0.0000,0.000


In [305]:
depot_psku_primary_df = depot_psku_primary_df.groupby(['chain', 'parent_material_code', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum']].sum().reset_index()

In [306]:
depot_psku_primary_df

,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000
1,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000
2,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186
3,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666
4,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632
...,...,...,...,...,...,...,...
5387,Zepto,811279,2026-07-31,0.000,0.321778,0.3156,0.000
5388,Zepto,811287,2026-04-30,26.400,114.301509,0.0000,0.000
5389,Zepto,811287,2026-05-31,60.000,157.918621,122.3085,0.000
5390,Zepto,811287,2026-06-30,21.600,161.147344,179.2152,21.600


In [307]:
depot_psku_primary_df['channel'] = 'Qcom'
depot_psku_primary_df_qcom = depot_psku_primary_df.copy()

In [308]:
depot_psku_primary_query = """
SELECT
    CASE 
        WHEN MCM.chain = 'Big basket B2C' THEN 'Big Basket'
        WHEN MCM.chain = 'Flipkart-National' THEN 'Flipkart National'
        WHEN MCM.chain = 'Flipkart-Minutes' THEN 'Flipkart National'
        WHEN MCM.chain = 'Nykaa' THEN 'Nykaa'
        WHEN MCM.chain = 'Purplle' THEN 'Purplle'
        WHEN MCM.chain = 'Myntra' THEN 'Myntra'
        WHEN MCM.chain = 'MYNTRA' THEN 'Myntra'
        WHEN MCM.chain = 'Amazon B2C' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'ARIPL' THEN 'Amazon ARIPL'
        WHEN MCM.chain = 'RK WORLDINFOCOM' THEN 'Amazon RK'
        WHEN MCM.chain = 'RKWorld' THEN 'Amazon RK'
        WHEN MCM.chain = 'Flipkart-Grocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'FlipkartGrocery' THEN 'Flipkart Grocery'
        WHEN MCM.chain = 'Firstcry' THEN 'First Cry'
        WHEN MCM.chain = 'CITIMALL' THEN 'City Mall'
        ELSE MCM.chain
    END AS chain,
    MM.parent_material_code,
    MM.material_group_code,
    MESR.month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN 
(
    SELECT
        customer,
        chain_type,
        chain
    FROM
        mst_chain_master
    WHERE
        chain_type = 'E Com B2C' AND 
        chain IN ('Flipkart-National', 'Flipkart-Grocery', 'Big basket B2C', 'RK WORLDINFOCOM', 'Amazon B2C', 
        'FATEHPURIA HYGIENE', 'Nykaa', 'Flipkart-Minutes', 'Purplle', 'Myntra', 'Dealshare', 'FlipkartGrocery', 'City Mall', '1MG', 
        'ARIPL', 'First Cry', 'Meesho', 'RKWorld', 'CITIMALL', 'Firstcry', 'EMAZING DEALS', 'MYNTRA')
) MCM ON MESR.distributor_code = MCM.customer
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
WHERE
    MESR.month_date between '2025-12-31' and '2026-07-31'
GROUP BY 1, 2, 3, 4
ORDER BY 1, 3, 2, 4
"""
depot_psku_primary_df = pd.read_sql(
    depot_psku_primary_query,
    prod_conn
)
depot_psku_primary_df.columns = depot_psku_primary_df.columns.str.lower()
depot_psku_primary_df['parent_material_code'] = depot_psku_primary_df['parent_material_code'].astype(int)
depot_psku_primary_df['month_date'] = pd.to_datetime(depot_psku_primary_df['month_date'])
depot_psku_primary_df = realign_pskus(depot_psku_primary_df.copy(), 'parent_material_code', channel='ECOM')
depot_psku_primary_df = depot_psku_primary_df.groupby(
    ['chain', 'parent_material_code', 'material_group_code', 'month_date'], as_index=False, dropna=False
).sum()
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['parent_material_code'] != 715096]
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()
for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    if depot_psku_primary_df[col].min() < 0:
        print(col)

for col in ['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']:
    depot_psku_primary_df[col] = depot_psku_primary_df[col].clip(lower=0)

pri_actuals_vol_rum
sec_actuals_vol_rum


In [309]:
depot_psku_primary_df = depot_psku_primary_df[depot_psku_primary_df['material_group_code']!='PABABY_SP']
depot_psku_primary_df.duplicated(subset=['chain', 'parent_material_code', 'month_date']).sum()


27

In [310]:
depot_psku_primary_df = depot_psku_primary_df.groupby(['chain', 'parent_material_code', 'month_date'])[['pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum']].sum().reset_index()

In [311]:
depot_psku_primary_df['channel'] = 'Ecom'
depot_psku_primary_df_ecom = depot_psku_primary_df.copy()

In [312]:
depot_psku_primary_df = pd.concat([depot_psku_primary_df_qcom, depot_psku_primary_df_ecom], ignore_index=True)
depot_psku_primary_df

,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,Blinkit,718287,2026-06-30,0.000,1.852098,0.2275,0.000,Qcom
1,Blinkit,718287,2026-07-31,0.000,2.175176,0.0000,0.000,Qcom
2,Blinkit,718288,2025-12-31,0.000,0.000000,76.6195,66.186,Qcom
3,Blinkit,718288,2026-01-31,51.666,49.549496,52.9887,51.666,Qcom
4,Blinkit,718288,2026-02-28,61.632,55.342992,60.4012,61.632,Qcom
...,...,...,...,...,...,...,...,...
97611,Purplle,811287,2026-07-31,0.000,0.000000,17.9378,0.000,Ecom
97612,Purplle,811416,2026-06-01,0.000,0.176361,0.0000,0.000,Ecom
97613,Purplle,811416,2026-06-30,0.000,0.000000,0.6841,0.000,Ecom
97614,Purplle,811416,2026-07-01,0.000,0.035743,0.0000,0.000,Ecom


In [313]:
depot_psku_primary_df_ecom

,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,channel
0,1MG,718287,2026-07-01,0.0,0.004330,0.0000,0.0,Ecom
1,1MG,718288,2026-01-01,0.0,1.014738,0.0000,0.0,Ecom
2,1MG,718288,2026-02-01,0.0,0.861434,0.0000,0.0,Ecom
3,1MG,718288,2026-03-01,0.0,0.892977,0.0000,0.0,Ecom
4,1MG,718288,2026-04-01,0.0,4.790505,0.0000,0.0,Ecom
...,...,...,...,...,...,...,...,...
92219,Purplle,811287,2026-07-31,0.0,0.000000,17.9378,0.0,Ecom
92220,Purplle,811416,2026-06-01,0.0,0.176361,0.0000,0.0,Ecom
92221,Purplle,811416,2026-06-30,0.0,0.000000,0.6841,0.0,Ecom
92222,Purplle,811416,2026-07-01,0.0,0.035743,0.0000,0.0,Ecom


In [314]:
actuals_df = depot_psku_primary_df.copy()

In [315]:
actuals_df.duplicated(subset=['channel','chain', 'parent_material_code','month_date']).sum()

0

In [316]:
actuals_df = actuals_df.groupby(['channel','chain', 'parent_material_code',
        'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,channel,chain,parent_material_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,Ecom,1MG,718287,2026-07-01,0.0,0.004330,0.0000,0.0
1,Ecom,1MG,718288,2026-01-01,0.0,1.014738,0.0000,0.0
2,Ecom,1MG,718288,2026-02-01,0.0,0.861434,0.0000,0.0
3,Ecom,1MG,718288,2026-03-01,0.0,0.892977,0.0000,0.0
4,Ecom,1MG,718288,2026-04-01,0.0,4.790505,0.0000,0.0
...,...,...,...,...,...,...,...,...
97611,Qcom,Zepto,811279,2026-07-31,0.0,0.321778,0.3156,0.0
97612,Qcom,Zepto,811287,2026-04-30,26.4,114.301509,0.0000,0.0
97613,Qcom,Zepto,811287,2026-05-31,60.0,157.918621,122.3085,0.0
97614,Qcom,Zepto,811287,2026-06-30,21.6,161.147344,179.2152,21.6


In [317]:
actuals_df.columns

Index(['channel', 'chain', 'parent_material_code', 'month_date',
       'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum',
       'sec_actuals_vol_rum'],
      dtype='object')

In [318]:
actuals_df = actuals_df.rename(columns={
    'parent_material_code': 'psku',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol',
    'month_date': 'month'
})

In [319]:
actuals_df.head()

,channel,chain,psku,month,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol
0,Ecom,1MG,718287,2026-07-01,0.0,0.004330,0.0,0.0
1,Ecom,1MG,718288,2026-01-01,0.0,1.014738,0.0,0.0
2,Ecom,1MG,718288,2026-02-01,0.0,0.861434,0.0,0.0
3,Ecom,1MG,718288,2026-03-01,0.0,0.892977,0.0,0.0
4,Ecom,1MG,718288,2026-04-01,0.0,4.790505,0.0,0.0


In [ ]:
#qcom_df.rename(columns = {'Channel':'channel'}, inplace = True)

In [320]:
qcom_df['psku'] = qcom_df['psku'].astype(int)
actuals_df['psku'] = actuals_df['psku'].astype(int)

In [321]:
actuals_df['channel'] = actuals_df['channel'].str.upper()
qcom_df['channel'] = qcom_df['channel'].str.upper()
actuals_df['chain'] = actuals_df['chain'].str.lower()
qcom_df['chain'] = qcom_df['chain'].str.lower()


In [ ]:
# qcom_df['channel'].unique()

array(['QCOM', 'ECOM'], dtype=object)

In [ ]:
# actuals_df = actuals_df[actuals_df['month_date'] == '2025-12-31']
# actuals_df

In [322]:
x = qcom_df.copy()

In [323]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    actuals_df.drop([  'pri_actuals_vol_rum', 'pri_apo_plan_vol_rum'], axis=1),
    on=['channel','chain', 'psku', 'month'],
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [324]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,CMX_WELPD,3014.000
1,2027-03-31,4700_BCPC,222.000
2,2027-03-31,CMX_PRTPD,3014.000
3,2027-03-31,PA_CN_HGO,488.152
4,2027-03-31,TRU_RAWDF,800.000


In [325]:
len_before_merge = len(qcom_df)
qcom_df = qcom_df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(qcom_df)
del len_before_merge

In [326]:
qcom_df.isna().sum()

month                          0
chain                          0
psku                           0
brand                          0
portfolio                      0
m month                        0
run_month                      0
calculated primary vol         0
channel                        0
Consensus Vol             118775
Actuals Vol               118775
Index Rate                     0
dtype: int64

In [327]:
qcom_df['Consensus Vol'] = qcom_df['Consensus Vol'].fillna(0)
qcom_df['Actuals Vol'] = qcom_df['Actuals Vol'].fillna(0)

In [328]:
qcom_df['Consensus Vol'].max()

45691.280000000006

In [329]:
qcom_df.rename(columns={'calculated primary vol': 'Stat Vol'}, inplace=True)
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
152443,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337
152444,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,0.000,260000.000000
152445,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072
152446,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072


In [330]:
qcom_df['Stat Val'] = qcom_df['Stat Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Consensus Val'] = qcom_df['Consensus Vol'] * qcom_df['Index Rate'] / (10 ** 7)
qcom_df['Actuals Val'] = qcom_df['Actuals Vol'] * qcom_df['Index Rate'] / (10 ** 7)

In [331]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,Stat Vol,channel,Consensus Vol,Actuals Vol,Index Rate,Stat Val,Consensus Val,Actuals Val
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,66.186,138865.260689,1.063979,1.063979,0.919094
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,349274.001420,0.000000,0.000000,0.000000
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,0.000,230000.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152443,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,1712.605337,0.000000,0.000000,0.000000
152444,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,0.000,260000.000000,0.004801,0.000000,0.000000
152445,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000
152446,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,0.000,12860.631072,0.000000,0.000000,0.000000


In [335]:
qcom_df[(qcom_df['month'] == '2026-07-31') & (qcom_df['channel']=='ECOM') & (qcom_df['run_month']=='2026-06-30')]['Actuals Val'].sum()

41.56523905349086

In [336]:
qcom_df['Stat Error'] = qcom_df['Stat Val'] - qcom_df['Actuals Val']
qcom_df['Consensus Error'] = qcom_df['Consensus Val'] - qcom_df['Actuals Val']

qcom_df['Stat Abs Error'] = np.abs(qcom_df['Stat Error'])
qcom_df['Consensus Abs Error'] = np.abs(qcom_df['Consensus Error'])

In [337]:
qcom_df['stat_bias'] = qcom_df['Stat Error']/qcom_df['Actuals Val']
qcom_df = qcom_df.fillna(0)

import numpy as np
import pandas as pd

qcom_df['stat_bias'] = (
    qcom_df['stat_bias']
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

bins = [-np.inf, -0.15, -0.10, -0.05, 0, 0.05, 0.10, 0.15, np.inf]
labels = [
    '< -15%',
    '-15% to -10%',
    '-10% to -5%',
    '-5% to 0%',
    '0% to 5%',
    '5% to 10%',
    '10% to 15%',
    '> 15%'
]

qcom_df['stat_bias_bucket'] = pd.cut(
    qcom_df['stat_bias'],
    bins=bins,
    labels=labels,
    right=False  
)


In [ ]:
# delivery_df = pd.read_csv('/data/aman_singh/acuuracy_check/Export View of Month Vol & APO.csv')

# delivery_df['Channel'] = delivery_df['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df = delivery_df[delivery_df['Channel'] == 'QCOM']
# delivery_df = delivery_df.groupby(['Depot','PSKU'])['delivery_vol'].sum().reset_index(
# )#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})

# len_before_merge = len(qcom_df)
# qcom_df = qcom_df.merge(
#     delivery_df,
#     on=['Depot', 'PSKU'],
#     how='left'
# )
# assert len_before_merge == len(qcom_df)
# del len_before_merge

# qcom_df['Delivery Val'] = qcom_df['delivery_vol'] * qcom_df['Index Rate'] / (10 ** 7)

# qcom_df['Dp Error'] = qcom_df['Delivery Val'] - qcom_df['Actuals Val']

# qcom_df['Dp Abs Error'] = np.abs(qcom_df['Dp Error'])



In [338]:
qcom_df.columns = qcom_df.columns.str.lower()
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,index rate,stat val,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,...,138865.260689,1.063979,1.063979,0.919094,0.144885,0.144885,0.144885,0.144885,0.157639,> 15%
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,349274.001420,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,230000.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152443,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,...,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
152444,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,...,260000.000000,0.004801,0.000000,0.000000,0.004801,0.000000,0.004801,0.000000,0.000000,0% to 5%
152445,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,...,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%
152446,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,...,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%


In [ ]:
# df = pd.read_excel("/data/aman_singh/acuuracy_check/acc_framework_feb.xlsx")
# df.columns = df.columns.str.lower()
# df = df[df['forecast_granularity']!='Depot x PSKU']

In [ ]:
# df.columns = df.columns.str.lower()

In [339]:
qcom_df.columns

Index(['month', 'chain', 'psku', 'brand', 'portfolio', 'm month', 'run_month',
       'stat vol', 'channel', 'consensus vol', 'actuals vol', 'index rate',
       'stat val', 'consensus val', 'actuals val', 'stat error',
       'consensus error', 'stat abs error', 'consensus abs error', 'stat_bias',
       'stat_bias_bucket'],
      dtype='object')

In [340]:

qcom_df['forecast_granularity'] = 'Chain x PSKU'
qcom_df['forecast_type'] = 'offtakes_to_primary'
#qcom_df['channel'] = 'QCOM'
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,2025-12-31,blinkit,718287,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1,2025-12-31,blinkit,718288,SAFF GOLD,Saffola Oils,M,2025-12-31,76.619500,QCOM,76.6195,...,1.063979,0.919094,0.144885,0.144885,0.144885,0.144885,0.157639,> 15%,Chain x PSKU,offtakes_to_primary
2,2025-12-31,blinkit,718297,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
3,2025-12-31,blinkit,718299,PCNO(R),CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
4,2025-12-31,blinkit,718300,PCNO FLEX,CNO,M,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
152443,2026-09-30,zepto,811169,SW_SGPRF,Male Grooming,M+3,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
152444,2026-09-30,zepto,811181,SAF_CDPRS,Saffola Oils,M+3,2026-06-30,0.184668,QCOM,0.0000,...,0.000000,0.000000,0.004801,0.000000,0.004801,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
152445,2026-09-30,zepto,811267,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
152446,2026-09-30,zepto,811269,PA_ESS_HO,Hair Oils,M+3,2026-06-30,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary


In [341]:
qcom_df = qcom_df[qcom_df['m month'] == 'M+1']

In [342]:
final_df

,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,psku,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734910,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
1,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734988,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
2,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734993,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
3,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,734995,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
4,Depot x PSKU,secondary,B2B,0,JH_FRG_L,2026-05-31,M+1,2026-06-30,D117,735043,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0% to 5%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
624591,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811267,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624592,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811268,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624593,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-06-30,M+1,2026-07-31,D677,811269,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%
624594,Depot x PSKU,offtakes_to_primary,ECOM,Saffola Oils,SAF_CDPRS,2026-06-30,M+1,2026-07-31,D677,811279,...,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN,0.0,0% to 5%


In [220]:
df_offtakes

,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,offtakes_forecasted_value,...,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,0.421023,...,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,0.000000,...,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,0.187607,...,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,0.000000,...,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,0.113964,...,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20595,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,...,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
20596,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.002372,...,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
20597,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,0.000524,...,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
20598,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,0.000000,...,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [221]:
qcom_df

,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,consensus vol,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
1571,2026-01-31,blinkit,718287,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1572,2026-01-31,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2025-12-31,52.041436,QCOM,52.9887,...,0.735829,0.717461,0.005213,0.018368,0.005213,0.018368,0.007267,0% to 5%,Chain x PSKU,offtakes_to_primary
1573,2026-01-31,blinkit,718297,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1574,2026-01-31,blinkit,718299,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1575,2026-01-31,blinkit,718300,PCNO FLEX,CNO,M+1,2025-12-31,0.000000,QCOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
104174,2026-06-30,purplle,811181,SAF_CDPRS,Saffola Oils,M+1,2026-05-31,0.000000,ECOM,0.0638,...,0.001659,0.000000,0.000000,0.001659,0.000000,0.001659,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
104175,2026-06-30,purplle,811267,PA_ESS_HO,Hair Oils,M+1,2026-05-31,0.000000,ECOM,0.7664,...,0.000986,0.000000,0.000000,0.000986,0.000000,0.000986,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
104176,2026-06-30,purplle,811268,PA_ESS_HO,Hair Oils,M+1,2026-05-31,0.000000,ECOM,0.9587,...,0.001233,0.000000,0.000000,0.001233,0.000000,0.001233,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
104177,2026-06-30,purplle,811269,PA_ESS_HO,Hair Oils,M+1,2026-05-31,0.000000,ECOM,0.0000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary


In [ ]:
all_df = pd.concat([final_df,df_offtakes,qcom_df])

In [344]:
all_df[all_df['month'] == '2026-07-31'].to_excel('acc_framework_july.xlsx')

### The end

In [109]:
qcom_df.to_csv('/data/aman_singh/acuuracy_check/chain_psku_primary_accuracy_till_may.csv')

In [1]:
import pandas as pd
df = pd.read_excel('/data/aman_singh/acuuracy_check/acc_framework_may_final.xlsx', sheet_name = 'Sheet1')
df

,Unnamed: 0,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,...,stat error,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket
0,129118,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D673,...,0.022506,0.041685,0.022506,0.041685,0.0,0.0,0.0,0.0,0.201333,> 15%
1,129119,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D673,...,-0.004181,-0.002461,0.004181,0.002461,0.0,0.0,0.0,0.0,-0.132249,-15% to -10%
2,129120,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D674,...,0.003741,-0.002501,0.003741,0.002501,0.0,0.0,0.0,0.0,0.334012,> 15%
3,129121,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D674,...,-0.020124,0.019797,0.020124,0.019797,0.0,0.0,0.0,0.0,-0.223653,< -15%
4,129122,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30,M+1,2026-05-31,D676,...,0.074692,0.070734,0.074692,0.070734,0.0,0.0,0.0,0.0,0.218949,> 15%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38287,451102,Depot x PSKU,offtakes_to_primary,ECOM,Male Grooming,SW_SGPRF,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38288,451103,Depot x PSKU,offtakes_to_primary,ECOM,Saffola Oils,SAF_CDPRS,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38289,451104,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%
38290,451105,Depot x PSKU,offtakes_to_primary,ECOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,D677,...,0.000000,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%


In [2]:
qcom_df = pd.read_csv('/data/aman_singh/acuuracy_check/chain_psku_primary_accuracy_till_may.csv')
qcom_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,run_month,stat vol,channel,...,consensus val,actuals val,stat error,consensus error,stat abs error,consensus abs error,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,1571,2026-01-31,blinkit,718287,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
1,1572,2026-01-31,blinkit,718288,SAFF GOLD,Saffola Oils,M+1,2025-12-31,52.041436,QCOM,...,0.735829,0.717461,0.005213,0.018368,0.005213,0.018368,0.007267,0% to 5%,Chain x PSKU,offtakes_to_primary
2,1573,2026-01-31,blinkit,718297,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
3,1574,2026-01-31,blinkit,718299,PCNO(R),CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
4,1575,2026-01-31,blinkit,718300,PCNO FLEX,CNO,M+1,2025-12-31,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24588,86523,2026-05-31,zepto,810971,PA_ESS_HO,Hair Oils,M+1,2026-04-30,0.180968,QCOM,...,0.000102,0.011112,-0.010879,-0.011010,0.010879,0.011010,-0.979055,< -15%,Chain x PSKU,offtakes_to_primary
24589,86524,2026-05-31,zepto,811005,PA_ESS_HO,Hair Oils,M+1,2026-04-30,0.154828,QCOM,...,0.007044,0.003889,-0.003690,0.003155,0.003690,0.003155,-0.948800,< -15%,Chain x PSKU,offtakes_to_primary
24590,86525,2026-05-31,zepto,811169,SW_SGPRF,Male Grooming,M+1,2026-04-30,0.000000,QCOM,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary
24591,86526,2026-05-31,zepto,811181,SAF_CDPRS,Saffola Oils,M+1,2026-04-30,0.043734,QCOM,...,0.046896,0.000000,0.001137,0.046896,0.001137,0.046896,0.000000,0% to 5%,Chain x PSKU,offtakes_to_primary


In [15]:
offtakes_df = pd.read_csv('/data/aman_singh/acuuracy_check/acc_offtakes_till_may.csv')
offtakes_df

,Unnamed: 0,month,platform_name,parent_material_code,brand,portfolio,run_month,m month,pred_prophet,pred_rf,...,vol_in_rum,Index Rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,2026-04-30,M+1,29.403313,27.76920,...,30.2040,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,2026-04-30,M+1,0.000000,0.00000,...,0.0000,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,2026-04-30,M+1,7.271279,16.05050,...,12.1500,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,2026-04-30,M+1,0.000000,0.00000,...,0.0000,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,2026-04-30,M+1,3.942144,6.99084,...,9.6561,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,2026-03-31,M+1,0.000000,0.00000,...,0.0000,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,...,1.7640,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,2026-03-31,M+1,0.000000,0.00000,...,0.5880,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,2026-03-31,M+1,0.000000,0.00000,...,0.0000,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [16]:
offtakes_df.rename(columns={'platform_name':'chain', 'parent_material_code':'psku'
                            ,'Index Rate':'index rate'}, inplace=True)
offtakes_df.drop(['vol_in_rum','run_month'], axis=1, inplace=True)
offtakes_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,M+1,29.403313,27.76920,0.421023,ECOM,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,M+1,0.000000,0.00000,0.000000,ECOM,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,M+1,7.271279,16.05050,0.187607,ECOM,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,M+1,0.000000,0.00000,0.000000,ECOM,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,M+1,3.942144,6.99084,0.113964,ECOM,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.002372,ECOM,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.000524,ECOM,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [20]:
offtakes_df['month'].unique()
offtakes_df = offtakes_df[offtakes_df['month'].isin(['2026-04-30', '2026-05-31'])]


In [21]:
final_df = pd.concat([df,qcom_df])
final_df

,Unnamed: 0,forecast_granularity,forecast_type,channel,portfolio,brand,run_month,m month,month,depot,...,consensus error,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket,chain
0,129118,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D673,...,0.041685,0.022506,0.041685,0.0,0.0,0.0,0.0,0.201333,> 15%,NaN
1,129119,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D673,...,-0.002461,0.004181,0.002461,0.0,0.0,0.0,0.0,-0.132249,-15% to -10%,NaN
2,129120,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D674,...,-0.002501,0.003741,0.002501,0.0,0.0,0.0,0.0,0.334012,> 15%,NaN
3,129121,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D674,...,0.019797,0.020124,0.019797,0.0,0.0,0.0,0.0,-0.223653,< -15%,NaN
4,129122,Depot x PSKU,secondary,GT,CNO,KERALA,2026-04-30 00:00:00,M+1,2026-05-31 00:00:00,D676,...,0.070734,0.074692,0.070734,0.0,0.0,0.0,0.0,0.218949,> 15%,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24588,86523,Chain x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,NaN,...,-0.011010,0.010879,0.011010,NaN,NaN,NaN,NaN,-0.979055,< -15%,zepto
24589,86524,Chain x PSKU,offtakes_to_primary,QCOM,Hair Oils,PA_ESS_HO,2026-04-30,M+1,2026-05-31,NaN,...,0.003155,0.003690,0.003155,NaN,NaN,NaN,NaN,-0.948800,< -15%,zepto
24590,86525,Chain x PSKU,offtakes_to_primary,QCOM,Male Grooming,SW_SGPRF,2026-04-30,M+1,2026-05-31,NaN,...,0.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,zepto
24591,86526,Chain x PSKU,offtakes_to_primary,QCOM,Saffola Oils,SAF_CDPRS,2026-04-30,M+1,2026-05-31,NaN,...,0.046896,0.001137,0.046896,NaN,NaN,NaN,NaN,0.000000,0% to 5%,zepto


In [11]:
x = final_df.copy()

In [22]:
offtakes_df

,Unnamed: 0,month,chain,psku,brand,portfolio,m month,pred_prophet,pred_rf,offtakes_forecasted_value,channel,index rate,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket,forecast_granularity,forecast_type
0,0,2026-05-31,amazon aripl,718288,SAFF GOLD,Saffola Oils,M+1,29.403313,27.76920,0.421023,ECOM,138865.260689,30.318787,0.419429,-0.001594,0.001594,-0.003800,-5% to 0%,Chain x PSKU,offtakes
1,1,2026-05-31,amazon aripl,718321,SAFF KO,Saffola Oils,M+1,0.000000,0.00000,0.000000,ECOM,168827.536176,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
2,2,2026-05-31,amazon aripl,718322,SAFF KO,Saffola Oils,M+1,7.271279,16.05050,0.187607,ECOM,168827.536176,11.112333,0.205125,0.017519,0.017519,0.085405,5% to 10%,Chain x PSKU,offtakes
3,3,2026-05-31,amazon aripl,718323,SF_IMV_MK,Foods,M+1,0.000000,0.00000,0.000000,ECOM,739265.904412,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
4,4,2026-05-31,amazon aripl,718328,SAFF KOCO,Saffola Oils,M+1,3.942144,6.99084,0.113964,ECOM,123636.889888,9.217630,0.119385,0.005421,0.005421,0.045409,0% to 5%,Chain x PSKU,offtakes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16426,16426,2026-04-30,nykaa,810605,KAYA_ML,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,1226.374229,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes
16427,16427,2026-04-30,nykaa,810673,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.002372,ECOM,12860.631072,1.844295,0.002269,-0.000103,0.000103,-0.045519,-5% to 0%,Chain x PSKU,offtakes
16428,16428,2026-04-30,nykaa,810674,PA_ESS_HO,Hair Oils,M+1,0.000000,0.00000,0.000524,ECOM,12860.631072,0.407243,0.000756,0.000232,0.000232,0.307410,> 15%,Chain x PSKU,offtakes
16429,16429,2026-04-30,nykaa,810738,PABABY_GM,Skin Care,M+1,0.000000,0.00000,0.000000,ECOM,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0% to 5%,Chain x PSKU,offtakes


In [25]:
# final_df = pd.concat([final_df,offtakes_df])
#final_df
final_df.to_csv('acc_till_may.csv')

In [24]:
final_df.isnull().sum()

Unnamed: 0                         0
forecast_granularity               0
forecast_type                      0
channel                            0
portfolio                          0
brand                              0
run_month                       8129
m month                            0
month                              0
depot                          32722
psku                               0
stat vol                        8129
prophet vol                    61110
rf_vol                         61110
prophet heuristic vol          61110
rf heuristic vol               61110
consensus vol                   8129
actuals vol                     8129
key                            61110
index rate                         0
stat val                        8129
consensus val                   8129
actuals val                     8129
stat error                      8129
consensus error                 8129
stat abs error                  8129
consensus abs error             8129
p

In [7]:
import pickle
with open('/data/aman_singh/acuuracy_check/prophet_models (2).pkl', 'rb') as f:
    model = pickle.load(f)

In [6]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 2.0,
 'n_changepoints': 4,
 'yearly_seasonality': 5}

In [8]:
model['ORS_D535_731588']

{'changepoint_prior_scale': 0.01,
 'changepoint_range': 0.8,
 'seasonality_prior_scale': 0.1,
 'n_changepoints': 4,
 'yearly_seasonality': 4}

In [2]:
print(model.growth)

AttributeError: 'dict' object has no attribute 'growth'